# 05 · ¿Qué pasa si ignoramos señales tras un mercado bajista?

Añadimos una alternativa sencilla antes del aprendizaje automático. El filtro observa los **24 meses anteriores** y decide si acepta la señal o permanece fuera. La regla de momentum o reversión sigue proponiendo la dirección original.

**Antes de empezar, repite los notebooks 01 y 02 con los supuestos indicados.** Ahora preparamos precios desde diciembre de 1924 para calcular el filtro desde enero de 1927. La evaluación sigue teniendo 756 meses hasta 1989; la historia adicional solo sirve para los primeros cálculos. No se utiliza el tramo desde 1990.


## 1. La regla y sus límites

Al decidir en el mes $m$ se calcula $P_m/P_{m-24}-1$. Si es negativo, el mes siguiente se permanece en efectivo. Si es cero o positivo, se sigue la señal original. Por ejemplo, si el precio pasa de 100 a 90 en esos dos años, el indicador es −10 % y el filtro rechaza la señal siguiente.

Es una adaptación del indicador de mercado bajista de [Daniel y Moskowitz (2016)](https://doi.org/10.1016/j.jfineco.2015.12.002). Usamos precios medios nominales de Shiller, sin dividendos como predictor, y un veto a las dos reglas. No replicamos su cartera transversal o su estrategia dinámica. Tampoco usamos el rebote del mes objetivo, que aún no se conocería, ni añadimos un umbral de volatilidad.

Se mantienen los costes, dividendos y remuneración del efectivo indicados en la siguiente celda. El ajuste por volatilidad se estudia por separado en el notebook 06.


In [ ]:
from tfm.data import project_root, save_result
from tfm.comparators import simulate_bear_comparison, BEAR_WARNINGS
from tfm.simulation import summarize_baselines, WARNINGS as SIMULATION_WARNINGS
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.data import load_monthly, read_result
from tfm.comparators import BEAR_PROTOCOL, BEAR_STRATEGIES

ROOT = project_root()
parametros = {"initial_capital": 100.0, "transaction_cost_rate": 0.001,
              "annual_borrow_rate": 0.01, "include_dividends": True}
mensuales = load_monthly(ROOT)
componentes, _ = read_result(ROOT, "monthly_return_components")
senales, _ = read_result(ROOT, "shiller_signals")
print(f"Historia preparada: {mensuales.period.iloc[0]} → {mensuales.period.iloc[-1]}")
print(f"Meses comparados: {componentes.period.iloc[0]} → {componentes.period.iloc[-1]} ({len(componentes)} meses)")
print("Filtro:", BEAR_PROTOCOL["formula"])
display(pd.DataFrame([parametros]).T.rename(columns={0: "Supuesto de la simulación"}))


## 2. Comparar las cinco cuentas

**Esta celda guarda o sustituye** `data/processed/bear_filter_simulations.parquet` y su JSON. No descarga fuentes ni modifica los archivos de las otras simulaciones. Repite las cuentas sin filtrar con las mismas entradas para poder compararlas exactamente.

Cada cuenta usa los mismos 756 meses, capital inicial y costes. La tabla añade los precios y fechas del indicador y las señales originales. El número de señales bloqueadas solo cuenta señales distintas de cero, momentum ya puede proponer efectivo por sí mismo.


In [ ]:
simulaciones = simulate_bear_comparison(componentes, senales, mensuales, parametros)
context = simulaciones.loc[simulaciones.strategy.eq("buy_and_hold")]
informe = {"status": "passed", "rows": len(simulaciones), "months_per_strategy": len(componentes),
          "first_period": componentes.period.iloc[0], "last_period": componentes.period.iloc[-1],
          "parameters": parametros, "bear_filter": BEAR_PROTOCOL,
          "bear_months": int(context.bear_market.sum()),
          "blocked_momentum_signals": int((context.bear_market & context.original_momentum_signal.ne(0)).sum()),
          "blocked_reversal_signals": int((context.bear_market & context.original_reversal_signal.ne(0)).sum()),
          "strategies": summarize_baselines(simulaciones, parametros, strategies=BEAR_STRATEGIES),
          "warnings": [*SIMULATION_WARNINGS, *BEAR_WARNINGS]}
resumen = save_result(ROOT, "bear_filter_simulations", simulaciones, informe)
print(resumen["path"])
print(f"{informe['rows']} filas: cinco cuentas × {informe['months_per_strategy']} meses")
print(f"Meses bajistas: {informe['bear_months']}")
print(f"Señales bloqueadas: momentum {informe['blocked_momentum_signals']}, reversión {informe['blocked_reversal_signals']}")
metricas = pd.DataFrame(informe["strategies"]).set_index("label")
vista = metricas[["annualized_return", "max_drawdown", "sharpe", "annualized_turnover"]].copy()
for columna in ["annualized_return", "max_drawdown"]:
    vista[columna] *= 100
vista = vista.rename(columns={"annualized_return": "Anualizada (%)", "max_drawdown": "Caída máxima (%)",
                              "sharpe": "Sharpe", "annualized_turnover": "Rotación anual (veces)"})
display(vista.round(4))


## 3. Entender mayo de 1975

Primero se muestra la ventana de precios que conocía la decisión de abril de 1975. Después aparecen las posiciones originales y filtradas para mayo. La fila de comprar y mantener es una referencia y no se somete al veto.

Una posición cero significa efectivo, que recibe RF. Pasar de una posición anterior a efectivo puede generar un coste de cierre. No equivale a borrar las pérdidas de operaciones pasadas ni a empezar con la misma cuenta que la estrategia sin filtrar, sus capitales ya pueden ser diferentes.


In [ ]:
MES_OBJETIVO = "1975-05"
mes = simulaciones.loc[simulaciones.period.eq(MES_OBJETIVO)].set_index("strategy")
if len(mes) != len(BEAR_STRATEGIES):
    raise ValueError("Elige un mes dentro del desarrollo preparado.")
contexto = mes.loc["buy_and_hold"]
display(pd.DataFrame([
    {"Dato": "Inicio de la ventana", "Valor": contexto.lookback_start_period},
    {"Dato": "Precio al inicio", "Valor": contexto.lookback_start_price},
    {"Dato": "Mes de decisión", "Valor": contexto.decision_period},
    {"Dato": "Precio al decidir", "Valor": contexto.decision_price},
    {"Dato": "Variación de 24 meses (%)", "Valor": 100 * contexto.market_return_24m},
    {"Dato": "¿Se activa el veto bajista?", "Valor": bool(contexto.bear_market)},
]))
vista_mes = pd.DataFrame({"Cuenta": mes.index.map(BEAR_STRATEGIES), "Posición": mes.position.to_numpy(),
                         "Neta del mes (%)": 100 * mes.net_return.to_numpy(),
                         "Coste de negociación": mes.transaction_cost.to_numpy(),
                         "Capital al terminar": mes.capital_end.to_numpy()}).set_index("Cuenta")
display(vista_mes.round(6))


## 4. Evolución y caídas

Cada fila de gráficos compara una regla con y sin filtro, usando comprar y mantener como referencia. El capital incluye el valor inicial antes del primer mes. Por defecto se usa escala logarítmica para poder ver las cuentas con evoluciones muy distintas, se puede desactivar sin cambiar los resultados.


In [ ]:
ESCALA_LOGARITMICA = True
evolucion = simulaciones.pivot(index="period", columns="strategy", values="capital_end")
mes_inicial = str(pd.Period(evolucion.index[0], freq="M") - 1)
inicial = pd.DataFrame({nombre: [parametros["initial_capital"]] for nombre in BEAR_STRATEGIES}, index=[mes_inicial])
evolucion = pd.concat([inicial, evolucion]).sort_index()
fechas = pd.PeriodIndex(evolucion.index, freq="M").to_timestamp(how="end")
caidas = 100 * (evolucion / evolucion.cummax() - 1)
fig, ejes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, layout="constrained")
for fila, regla in enumerate(["momentum", "reversal"]):
    for nombre, estilo in zip(["buy_and_hold", regla, f"{regla}_bear"], [":", "--", "-"]):
        ejes[fila, 0].plot(fechas, evolucion[nombre], linestyle=estilo, label=BEAR_STRATEGIES[nombre])
        ejes[fila, 1].plot(fechas, caidas[nombre], linestyle=estilo, label=BEAR_STRATEGIES[nombre])
    if ESCALA_LOGARITMICA:
        ejes[fila, 0].set_yscale("log")
    ejes[fila, 0].set(title=BEAR_STRATEGIES[regla], ylabel="Capital normalizado")
    ejes[fila, 1].set(ylabel="Caída desde máximos (%)")
    for eje in ejes[fila]:
        eje.grid(alpha=0.25)
        eje.legend(fontsize=8)
for eje in ejes[-1]:
    eje.set_xlabel("Año")
fig.suptitle("Filtro bajista · desarrollo 1927–1989")
plt.show()


## 5. Qué podemos concluir

Esta comparación mide el efecto de una regla sencilla y declarada de antemano. Una diferencia de Sharpe o de caída no demuestra que el filtro sepa anticipar el mercado, también cambia exposición, tiempo en efectivo, dividendos y costes.

La regla se aplica tanto a momentum como a reversión para disponer de una referencia explícita para cada una. Su uso en reversión no está justificado automáticamente por un artículo sobre momentum transversal; es una adaptación que debemos contrastar. Se mantienen las limitaciones de los promedios de Shiller y de los dividendos reconstruidos.

No cambiaremos la ventana o el veto después para favorecer los resultados. Quedan el ajuste por volatilidad y, después, las variables, las etiquetas y la validación de los modelos. La evaluación desde 1990 sigue deshabilitada.
